# FedTrace — 04: Grant Outlays

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fedtrace/fedtrace.github.io/blob/main/notebooks/04_grant_outlays.ipynb)

**Runtime:** CPU only  
**Estimated time:** ~30 min (first run) — resumable if interrupted  
**Input checkpoints:** `data/raw/02_grants.jsonl` — written by notebook 02  
**Publishes to GitHub:** `data/raw/04_grant_outlays.jsonl`, `data/outputs/04_grant_outlays.json`, `data/findings/04_grant_outlays.md`

**Driving questions:**
1. Does the USASpending `spending_by_award` search endpoint return `Total Outlays` for assistance awards when queried by FAIN?
2. If so, how do we extract the FAIN from the `generated_unique_award_id` format reliably?
3. After incorporating outlay data, what fraction of matched grants have a complete three-number record?
4. What are the aggregate outlay totals for cancelled grants?

**Investigation log:**
- `/api/v2/awards/{id}/` (notebook 02): `total_outlays` field present but genuinely null for all 12,361 assistance awards.
- `/api/v2/awards/funding/` (Section 2, Probe A): HTTP 200, zero result rows for all sampled grants. Endpoint does not index assistance awards.
- `/api/v2/search/spending_by_award/` with FAIN (Section 3, Probe B): **under investigation.** This is the same endpoint that returns `Total Outlays` for contracts — testing whether it works for assistance awards with FAIN as the award identifier.

**Design:** probe first, then fetch. Section 3 tests the search endpoint on 10 grants. Section 4 runs the full fetch only if the probe confirms outlay data is present.

Current findings: [`data/findings/04_grant_outlays.md`](../data/findings/04_grant_outlays.md).

## Setup

Run cells 1–4 at the start of every session. They are idempotent — safe to re-run.

In [ ]:
# ── CELL 1: Clone repo & install dependencies ─────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/fedtrace.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/fedtrace/fedtrace.github.io.git', str(REPO)],
        check=True, env=env,
    )
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('04_grant_outlays', requirements_path=str(REPO / 'requirements.txt'))

In [ ]:
# ── CELL 2: Pull latest & set up paths ────────────────────────────────────────
from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
PATHS['raw_dir'].mkdir(parents=True, exist_ok=True)
PATHS['outputs_dir'].mkdir(parents=True, exist_ok=True)
(PATHS['data_root'] / 'findings').mkdir(parents=True, exist_ok=True)
print(f'Repo ready at {REPO}')

In [ ]:
import json
import re
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

USA_BASE = 'https://api.usaspending.gov/api/v2'
DELAY    = 0.25
BATCH    = 50

# Assistance award type codes grouped by API constraint.
# spending_by_award requires all codes in one request to belong to the same group.
ASSIST_TYPE_GROUPS = {
    'grants':          ['02', '03', '04', '05', '06', '09'],
    'direct_payments': ['10', '11'],
    'loans':           ['07', '08'],
}

# Fields to request from the search endpoint
SEARCH_FIELDS = [
    'Award ID', 'Recipient Name', 'Award Amount', 'Total Outlays',
    'Description', 'Awarding Agency',
    'Period of Performance Start Date',
    'Period of Performance Current End Date',
    'generated_internal_id',
]

_RETRYABLE = (
    requests.exceptions.ConnectionError,
    requests.exceptions.Timeout,
    requests.exceptions.ChunkedEncodingError,
)


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(
            f'Checkpoint not found: {path}\n'
            'Run notebook 02 to completion before running this notebook.'
        )
    records = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_jsonl_ids(path: Path, id_field: str) -> set:
    path = Path(path)
    if not path.exists():
        return set()
    ids = set()
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                ids.add(json.loads(line).get(id_field))
    return ids


def append_jsonl(path: Path, records: list[dict]) -> None:
    with open(path, 'a') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')


def extract_fain(generated_unique_award_id: str) -> str | None:
    """Extract FAIN from generated_unique_award_id.

    Format: ASST_{TYPE}_{FAIN}_{AGENCY_CODE}
    Example: ASST_NON_NA18OAR4320123_1330 -> NA18OAR4320123

    The agency code is the last underscore-delimited segment (4 alphanumeric chars).
    The type is the second segment (e.g. NON, AGG).
    Everything between is the FAIN — joined back with underscores if the FAIN itself
    contains underscores.
    """
    parts = generated_unique_award_id.split('_')
    if len(parts) < 4 or parts[0] != 'ASST':
        return None
    # parts[0]='ASST', parts[1]=TYPE, parts[2:-1]=FAIN segments, parts[-1]=AGENCY_CODE
    fain_parts = parts[2:-1]
    return '_'.join(fain_parts) if fain_parts else None

In [ ]:
# ── CELL 4: Paths ─────────────────────────────────────────────────────────────
grants_path        = PATHS['raw_dir'] / '02_grants.jsonl'
grant_outlays_path = PATHS['raw_dir'] / '04_grant_outlays.jsonl'
output_json_path   = PATHS['outputs_dir'] / '04_grant_outlays.json'
findings_md_path   = PATHS['data_root'] / 'findings' / '04_grant_outlays.md'

print('Input checkpoint state:')
n = sum(1 for l in grants_path.read_text().splitlines() if l.strip()) if grants_path.exists() else 0
status = 'OK' if grants_path.exists() else 'MISSING — run notebook 02'
print(f'  grants:         {n:,} records  [{status}]')

n_fetched = len(load_jsonl_ids(grant_outlays_path, 'award_id'))
print(f'  grant_outlays:  {n_fetched:,} records fetched so far')

## 1. Load Grant Award IDs

In [ ]:
grants_records = load_jsonl(grants_path)
grants_df      = pd.DataFrame(grants_records)

print(f'Grants loaded: {len(grants_df):,}')
print()

# Confirm the known gap
if 'total_outlays' in grants_df.columns:
    null_count = grants_df['total_outlays'].isna().sum()
    print(f'total_outlays from /awards/{{id}}/ endpoint: {null_count:,} null ({null_count/len(grants_df)*100:.1f}%)')

award_ids = grants_df['award_id'].dropna().tolist() if 'award_id' in grants_df.columns else []

# Extract FAINs and validate extraction coverage
grants_df['fain'] = grants_df['award_id'].apply(
    lambda x: extract_fain(str(x)) if pd.notna(x) else None
)
n_fain_ok = grants_df['fain'].notna().sum()
n_fain_fail = grants_df['fain'].isna().sum()
print(f'FAIN extraction: {n_fain_ok:,} ok, {n_fain_fail:,} failed')
if n_fain_fail > 0:
    print('Failed award_id samples:')
    print(grants_df[grants_df['fain'].isna()]['award_id'].head(5).tolist())

fains = grants_df['fain'].dropna().tolist()
# Build reverse map: FAIN -> award_id (for joining results back)
fain_to_award_id = dict(zip(grants_df['fain'].dropna(), grants_df.loc[grants_df['fain'].notna(), 'award_id']))

print(f'\nSample award_id -> FAIN:')
for aid, fain in list(zip(award_ids[:5], fains[:5])):
    print(f'  {aid} -> {fain}')

## 2. Probe A — Confirmed Dead End: `/awards/funding/`

**Result (run 2026-03-30):** `POST /api/v2/awards/funding/` returns HTTP 200 with zero result rows for all sampled grants. The endpoint does not index assistance awards. This path is closed.

The cell below is kept for reproducibility — re-running it will confirm the same result.

In [ ]:
PROBE_A_N = 5  # Reduced — already confirmed dead end; keeping for reproducibility
probe_a_ids = award_ids[:PROBE_A_N]

print(f'Probe A: POST /api/v2/awards/funding/ ({PROBE_A_N} samples)')
print('Expected: HTTP 200, 0 result rows (confirmed dead end for assistance awards)')
print()

probe_a_results = []
for award_id in probe_a_ids:
    try:
        r = requests.post(f'{USA_BASE}/awards/funding/', json={'award_id': award_id, 'limit': 10, 'page': 1}, timeout=15)
        data = r.json() if r.status_code == 200 else {}
        row_count = len(data.get('results', []))
        probe_a_results.append({'award_id': award_id, 'status': r.status_code, 'rows': row_count})
        print(f'  {award_id}: HTTP {r.status_code}, {row_count} rows')
    except Exception as e:
        probe_a_results.append({'award_id': award_id, 'status': None, 'rows': None})
        print(f'  {award_id}: Error — {e}')
    time.sleep(DELAY)

has_rows = any(r['rows'] and r['rows'] > 0 for r in probe_a_results)
print(f'\nResult: {"rows present — re-evaluate" if has_rows else "confirmed dead end (0 rows for all samples)"}')

## 3. Probe B — `spending_by_award` Search Endpoint with FAIN

The `spending_by_award` endpoint is what returns `Total Outlays` for contracts (notebook 02). For contracts, we pass PIIDs in the `award_ids` filter. For assistance awards, the equivalent identifier is the FAIN (Federal Award Identification Number).

This probe:
1. Extracts FAINs from the `generated_unique_award_id` values in the grants checkpoint
2. Submits a batch of 10 FAINs to `spending_by_award` with assistance type codes
3. Checks whether `Total Outlays` is populated in the results
4. Cross-checks `Award Amount` against the known `total_obligation` from notebook 02 to confirm the search matched the right award

In [ ]:
PROBE_B_N = 10
probe_b_fains = fains[:PROBE_B_N]

print(f'Probe B: spending_by_award with FAINs ({PROBE_B_N} samples)')
print(f'Award type groups: {list(ASSIST_TYPE_GROUPS.keys())}')
print(f'Sample FAINs: {probe_b_fains[:5]}')
print()

results = []

for group_name, type_codes in ASSIST_TYPE_GROUPS.items():
    payload = {
        'filters': {
            'award_ids':        probe_b_fains,
            'award_type_codes': type_codes,
        },
        'fields': SEARCH_FIELDS,
        'page': 1,
        'limit': PROBE_B_N * 2,
    }
    try:
        r = requests.post(f'{USA_BASE}/search/spending_by_award/', json=payload, timeout=30)
        print(f'{group_name} {type_codes}: HTTP {r.status_code}', end='')
        if r.status_code == 200:
            group_results = r.json().get('results', [])
            results.extend(group_results)
            print(f', {len(group_results)} results')
        else:
            print(f', response: {r.text[:200]}')
    except Exception as e:
        print(f'{group_name}: Error — {e}')
    time.sleep(DELAY)

print(f'\nTotal results across all groups: {len(results)}')

if results:
    print(f'\nFields in result[0]: {list(results[0].keys())}')

    outlays_vals = [r.get('Total Outlays') for r in results]
    n_non_null   = sum(1 for v in outlays_vals if v is not None)
    n_nonzero    = sum(1 for v in outlays_vals if v is not None and float(v) != 0)
    print(f'\nTotal Outlays: {n_non_null}/{len(results)} non-null, {n_nonzero}/{len(results)} non-zero')
    if n_non_null > 0:
        print(f'Sample values: {[v for v in outlays_vals if v is not None][:5]}')

    print(f'\nCross-check Award Amount vs total_obligation from notebook 02:')
    for result in results[:5]:
        returned_id   = result.get('Award ID', '')
        award_id_back = fain_to_award_id.get(returned_id)
        nb02_row      = grants_df[grants_df['award_id'] == award_id_back] if award_id_back else pd.DataFrame()
        nb02_obl      = float(nb02_row['total_obligation'].values[0]) if not nb02_row.empty else None
        returned_amt  = result.get('Award Amount')
        match_ok      = (
            nb02_obl is not None and returned_amt is not None
            and abs(float(returned_amt) - nb02_obl) / (abs(nb02_obl) or 1) < 0.01
        )
        print(
            f'  {returned_id}: '
            f'search={returned_amt}, nb02={nb02_obl}, '
            f'match={"ok" if match_ok else "MISMATCH or not found"}'
        )
else:
    print('No results from any group — FAINs may not be valid identifiers.')


## 4. Full Fetch via Search Endpoint

**Run this section only if Probe B above confirmed that `Total Outlays` is populated for at least some grants.**

If the search endpoint returned results but `Total Outlays` is null for all of them, document the finding and stop — grant outlay data requires the bulk download path.

Strategy: batch FAINs in groups of 50, submit each batch to `spending_by_award`, save one record per matched grant to `data/raw/04_grant_outlays.jsonl`. Resumable — already-fetched award IDs are skipped.

**Note on matching:** the search endpoint returns `Award ID` which may be the FAIN (not the `generated_unique_award_id`). The join back to notebook 02 grants uses `fain_to_award_id`.

In [ ]:
# ── GATE: confirm Probe B showed Total Outlays before full fetch ───────────────
if 'results' not in dir() or results is None:
    raise RuntimeError('Run Probe B (Section 3) before this cell.')

n_with_outlays = sum(
    1 for r in results
    if r.get('Total Outlays') is not None
)

if not results:
    print('STOP: Probe B returned no results.')
    print('The search endpoint did not match any FAINs for assistance awards.')
    print('Next path: USASpending bulk download (Assistance_PrimeTransactions).')
    print('See CLAUDE.md open decisions for bulk download approach.')
    raise SystemExit(0)

if n_with_outlays == 0:
    print(f'STOP: Probe B matched {len(results)} grants but Total Outlays is null for all.')
    print('USASpending does not populate outlay data for assistance awards via any search endpoint.')
    print('Next path: USASpending bulk download (Assistance_PrimeTransactions).')
    print('See CLAUDE.md open decisions for bulk download approach.')
    raise SystemExit(0)

print(f'Probe B confirmed Total Outlays in {n_with_outlays}/{len(results)} sampled results.')
print('Proceeding with full fetch...')

In [ ]:
def _search_batch(fains_batch: list[str]) -> list[dict]:
    """Search spending_by_award for a batch of FAINs across all assistance type groups."""
    all_results = []
    for group_name, type_codes in ASSIST_TYPE_GROUPS.items():
        payload = {
            'filters': {
                'award_ids':        fains_batch,
                'award_type_codes': type_codes,
            },
            'fields': SEARCH_FIELDS,
            'page': 1,
            'limit': len(fains_batch) * 2,
        }
        for attempt in range(6):
            wait = 2 ** attempt
            try:
                r = requests.post(f'{USA_BASE}/search/spending_by_award/', json=payload, timeout=30)
                if r.status_code == 429:
                    time.sleep(wait)
                    continue
                r.raise_for_status()
                all_results.extend(r.json().get('results', []))
                break
            except _RETRYABLE as e:
                print(f'  Connection error [{group_name}] attempt {attempt+1}/6: {e}')
                time.sleep(wait)
        time.sleep(DELAY)
    return all_results


already_fetched = load_jsonl_ids(grant_outlays_path, 'award_id')
remaining = [
    (award_id, fain)
    for award_id, fain in zip(
        grants_df['award_id'].dropna(),
        grants_df['fain'].dropna(),
    )
    if award_id not in already_fetched and fain is not None
]

print(f'Total grants with FAIN: {len(fains):,}')
print(f'Already fetched:        {len(already_fetched):,}')
print(f'Remaining:              {len(remaining):,}')

skipped_batches = 0
matched_count   = 0
unmatched_count = 0

for i in tqdm(range(0, len(remaining), BATCH), desc='Grant outlays'):
    batch       = remaining[i:i+BATCH]
    batch_fains = [b[1] for b in batch]

    try:
        search_results = _search_batch(batch_fains)
    except RuntimeError:
        print(f'  Skipping batch {i//BATCH} after exhausted retries')
        skipped_batches += 1
        continue

    results_by_fain = {r['Award ID']: r for r in search_results}

    for award_id, fain in batch:
        result = results_by_fain.get(fain)
        if result:
            record = {
                'award_id':      award_id,
                'fain':          fain,
                'total_outlays': result.get('Total Outlays'),
                'award_amount':  result.get('Award Amount'),
                'source':        'usaspending_spending_by_award',
            }
            matched_count += 1
        else:
            record = {
                'award_id':      award_id,
                'fain':          fain,
                'total_outlays': None,
                'award_amount':  None,
                'source':        'unmatched',
            }
            unmatched_count += 1
        append_jsonl(grant_outlays_path, [record])

total_fetched = len(load_jsonl_ids(grant_outlays_path, 'award_id'))
print(f'\nFetch complete: {total_fetched:,} records saved')
print(f'  Matched:   {matched_count:,}')
print(f'  Unmatched: {unmatched_count:,}')
if skipped_batches:
    print(f'  Skipped batches: {skipped_batches} — re-run to retry')


## 5. Updated Grant Three-Number Record

Join the search endpoint outlays onto the grants from notebook 02.

In [ ]:
if not grant_outlays_path.exists():
    raise FileNotFoundError(
        f'{grant_outlays_path} not found.\n'
        'Run the full fetch in Section 4 before this cell.'
    )

outlays_records = load_jsonl(grant_outlays_path)
outlays_df      = pd.DataFrame(outlays_records)

print(f'Outlay records loaded: {len(outlays_df):,}')

if 'total_outlays' in outlays_df.columns:
    n_non_null = outlays_df['total_outlays'].notna().sum()
    n_nonzero  = (outlays_df['total_outlays'].fillna(0).astype(float) > 0).sum()
    n_matched  = (outlays_df.get('source', '') == 'usaspending_spending_by_award').sum()
    print(f'  total_outlays non-null: {n_non_null:,}')
    print(f'  total_outlays > 0:      {n_nonzero:,}')
    print(f'  matched via search:     {n_matched:,}')

# Join onto grants
merged = grants_df.merge(
    outlays_df[['award_id', 'total_outlays', 'source']].rename(columns={'total_outlays': 'search_outlays'}),
    on='award_id',
    how='left',
)

def _to_float(v):
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

merged['ceiling']   = merged['value'].apply(_to_float)
merged['obligated'] = merged['total_obligation'].apply(_to_float)
merged['outlays']   = merged['search_outlays'].apply(_to_float)

n_grants    = len(merged)
n_all_three = int((
    merged['ceiling'].notna() &
    merged['obligated'].notna() &
    merged['outlays'].notna()
).sum())

print(f'\nThree-number record — grants:')
print(f'  Total:             {n_grants:,}')
print(f'  Ceiling present:   {merged["ceiling"].notna().sum():,}')
print(f'  Obligated present: {merged["obligated"].notna().sum():,}')
print(f'  Outlays present:   {merged["outlays"].notna().sum():,}')
print(f'  All three:         {n_all_three:,} ({n_all_three/(n_grants or 1)*100:.1f}%)')

# Cross-check: search Award Amount vs nb02 total_obligation
if 'award_amount' in outlays_df.columns:
    check_df = grants_df.merge(
        outlays_df[['award_id', 'award_amount']],
        on='award_id', how='inner',
    )
    check_df['nb02_obl']    = check_df['total_obligation'].apply(_to_float)
    check_df['search_amt']  = check_df['award_amount'].apply(_to_float)
    both = check_df[check_df['nb02_obl'].notna() & check_df['search_amt'].notna()]
    if not both.empty:
        close = (abs(both['search_amt'] - both['nb02_obl']) / both['nb02_obl'].abs().replace(0, float('nan')) < 0.01).sum()
        print(f'\nCross-check (search Award Amount vs nb02 total_obligation):')
        print(f'  Within 1%: {close:,} / {len(both):,} ({close/len(both)*100:.1f}%)')

## 6. Coverage and Gap Analysis

In [ ]:
def _safe_pct(a, b):
    return a / (b or 1) * 100

def _sum_col(df, col):
    if col not in df.columns:
        return None
    s = pd.to_numeric(df[col], errors='coerce')
    return round(float(s.sum()), 0) if s.notna().any() else None


agg = {
    'total_ceiling':   _sum_col(merged, 'ceiling'),
    'total_obligated': _sum_col(merged, 'obligated'),
    'total_outlays':   _sum_col(merged, 'outlays'),
}

# Ceiling gap
gap_valid = merged[
    merged['ceiling'].notna() & merged['obligated'].notna() & (merged['ceiling'] > 0)
].copy()
gap_valid['gap_pct'] = (gap_valid['ceiling'] - gap_valid['obligated']) / gap_valid['ceiling'] * 100

gap_stats = {}
if not gap_valid.empty:
    gap_stats = {
        'n':              len(gap_valid),
        'median_gap_pct': round(float(gap_valid['gap_pct'].median()), 1),
        'mean_gap_pct':   round(float(gap_valid['gap_pct'].mean()),   1),
        'pct_gt_50':      round(_safe_pct((gap_valid['gap_pct'] > 50).sum(), len(gap_valid)), 1),
    }

# Near-zero obligation
n_near_zero = 0
if not gap_valid.empty:
    obl_rate    = gap_valid['obligated'] / gap_valid['ceiling'].replace(0, float('nan'))
    n_near_zero = int((obl_rate < 0.01).sum())

# Outlay rate
outlay_valid = merged[
    merged['outlays'].notna() & merged['obligated'].notna() & (merged['obligated'] > 0)
].copy()
outlay_stats = {}
if not outlay_valid.empty:
    outlay_valid['outlay_rate'] = outlay_valid['outlays'] / outlay_valid['obligated']
    outlay_stats = {
        'n':                     len(outlay_valid),
        'median_outlay_rate':    round(float(outlay_valid['outlay_rate'].median()), 3),
        'mean_outlay_rate':      round(float(outlay_valid['outlay_rate'].mean()),   3),
        'pct_gt_80pct_disbursed': round(
            _safe_pct((outlay_valid['outlay_rate'] > 0.8).sum(), len(outlay_valid)), 1
        ),
    }

print('=== Aggregate Totals — Grants ===')
print(f'  Ceiling:   ${(agg["total_ceiling"] or 0):,.0f}')
print(f'  Obligated: ${(agg["total_obligated"] or 0):,.0f}')
print(f'  Outlays:   ${(agg["total_outlays"] or 0):,.0f}')
print()
print(f'=== Three-Number Coverage ===')
print(f'  {n_all_three:,} / {n_grants:,} ({_safe_pct(n_all_three, n_grants):.1f}%)')
print()
if gap_stats:
    print(f'=== Ceiling Gap ===')
    print(f'  n={gap_stats["n"]:,}, median={gap_stats["median_gap_pct"]}%, mean={gap_stats["mean_gap_pct"]}%')
    print(f'  {gap_stats["pct_gt_50"]}% had >50% of ceiling unobligated at cancellation')
    print()
if outlay_stats:
    print(f'=== Outlay Rate (outlays / obligated) ===')
    print(f'  n={outlay_stats["n"]:,}, median={outlay_stats["median_outlay_rate"]:.1%}, mean={outlay_stats["mean_outlay_rate"]:.1%}')
    print(f'  {outlay_stats["pct_gt_80pct_disbursed"]}% had >80% of obligated disbursed')

## 7. Publish

Verify output above, then push to GitHub.

In [ ]:
output = {
    'source_notebooks': ['02_fetch'],
    'grants': {
        'total_assembled':        n_grants,
        'all_three_numbers':      n_all_three,
        'all_three_numbers_pct':  round(_safe_pct(n_all_three, n_grants), 1),
        'near_zero_obligation':   n_near_zero,
        'ceiling_gap':            gap_stats,
        'outlay_rate':            outlay_stats,
        'aggregate':              agg,
    },
    'outlay_source': {
        'endpoint':     'POST /api/v2/search/spending_by_award/',
        'filter':       'award_ids (FAINs extracted from generated_unique_award_id)',
        'type_codes':   ASSIST_TYPES,
        'field':        'Total Outlays',
    },
    'dead_ends_confirmed': [
        '/api/v2/awards/{id}/ — total_outlays field present but null for all assistance awards',
        '/api/v2/awards/funding/ — HTTP 200, zero result rows for all sampled assistance awards',
    ],
    'methodology_notes': [
        'ceiling = DOGE value field',
        'obligated = USASpending total_obligation from /api/v2/awards/{id}/',
        'outlays = Total Outlays from spending_by_award search endpoint (FAIN lookup)',
        'FAIN extracted from generated_unique_award_id: ASST_{TYPE}_{FAIN}_{AGENCY_CODE}',
        'grant coverage limited to ~78% of DOGE grants with a direct USASpending link',
    ],
}

output_json_path.write_text(json.dumps(output, indent=2))
print(json.dumps(output, indent=2))

In [ ]:
outlay_line = (
    f'  - outlays: ${(agg["total_outlays"] or 0):,.0f} '
    f'(median outlay rate {outlay_stats.get("median_outlay_rate", 0):.1%}, '
    f'{outlay_stats.get("pct_gt_80pct_disbursed", 0)}% of grants >80% disbursed)'
) if outlay_stats else '  - outlays: not available'

gap_line = (
    f'  - ceiling gap: median {gap_stats["median_gap_pct"]}%, '
    f'{gap_stats["pct_gt_50"]}% of grants had >50% unobligated at cancellation'
) if gap_stats else '  - ceiling gap: insufficient data'

findings_md = f"""# Findings — 04: Grant Outlays

*Input: {n_grants:,} matched grants (from notebook 02 checkpoint).*  
*Methodology: `notebooks/04_grant_outlays.ipynb`*

## Confirmed

- **Dead ends (confirmed):**
  - `/api/v2/awards/{{id}}/` — `total_outlays` field present but null for all 12,361 assistance awards.
  - `/api/v2/awards/funding/` — HTTP 200, zero result rows for all sampled assistance awards.
- **Working path:** `POST /api/v2/search/spending_by_award/` with FAINs (extracted from `generated_unique_award_id`) and assistance type codes returns `Total Outlays` for grants.
- **Three-number coverage — grants:** {n_all_three:,}/{n_grants:,} ({_safe_pct(n_all_three, n_grants):.1f}%) now have ceiling, obligated, and outlays present.
- **Aggregate totals — grants:**
  - ceiling:   ${(agg['total_ceiling'] or 0):,.0f}
  - obligated: ${(agg['total_obligated'] or 0):,.0f}
{outlay_line}
{gap_line}

**Methodology constraints carried forward:**
- FAIN extracted from `generated_unique_award_id` as the third through penultimate underscore-delimited segments. Pattern: `ASST_{{TYPE}}_{{FAIN}}_{{AGENCY_CODE}}`.
- Grant coverage is limited to the ~78% of DOGE grant records with a direct USASpending link. The remaining ~22% have no confirmed automated linkage path.
- `ceiling`, `obligated`, and `outlays` are not interchangeable. Any single number in isolation is incomplete.

## Open

- Linkage path for grants with no `link` host (~22% of DOGE grant records) — unresolved.
- Combined three-number record across contracts and grants — notebook 05 or an updated assembly notebook.
"""

findings_md_path.write_text(findings_md)
print(findings_md)

In [ ]:
from scripts.colab_utils import publish_artifacts

publish_artifacts(
    paths=[
        'data/raw/04_grant_outlays.jsonl',
        'data/outputs/04_grant_outlays.json',
        'data/findings/04_grant_outlays.md',
    ],
    message='Grant outlays via spending_by_award search endpoint',
    repo_dir=REPO,
)